# Let AI Handle the Mess

## Building Agentic AI for Data Cleaning and Analysis

**Presented by:** Ana Petrova & Elena Atanasovska

**Hosted by:** [​gr.ai.ce](https://devrev.ai/graice) × [DATA_FAIR](https://datafair.si/)

**Date:** February 25, 2026

---

## The Problem

Working with real-world data always starts with cleaning. 

**Manual work does not scale.** You cannot sit and review every column of every file when you have dozens of datasets.

**Fully automated scripts lose human judgment.** A script that drops all nulls might be wrong — those nulls could mean "no reviews yet" and should be filled with zero instead.

The sweet spot is somewhere in between: let the machine handle the repetitive work, but **pause and ask a human** when judgment is needed.

```
Automation ────────────────────────── Human judgment
     ^                                        ^
     |                                        |
  "Drop all nulls"                "These nulls mean 'no review yet',
                                   fill with zero instead"
```

This is what **agentic AI** gives us: a system that works autonomously through routine steps but knows when to stop and ask.

---

## What We're Building

An AI agent that:

1. **Loads** CSV files and infers column types (using rules + an LLM)
2. **Pauses** to ask you to approve or correct each inference
3. **Cleans** the data — duplicates, missing values, unnecessary columns
4. **Analyzes** the cleaned data across all files
5. **Visualizes** the results

Every decision point is a **human-in-the-loop** pause. The human stays in control.

By the end of this workshop you will understand:

- How to model a multi-step workflow as a **state machine**
- How **nodes** and **routing functions** control the flow
- How to you enable **human-in-the-loop** interaction
- How an LLM acts as an **interpreter** of natural language, not as a controller/generator


---

## The Data

We are using the **Inside Airbnb** dataset — open data about Airbnb listings in cities around the world.

> **Data source:** [Inside Airbnb](http://insideairbnb.com/) by Murray Cox.
> Licensed under Creative Commons CC0 1.0 Universal.

Each city folder contains a `listings.csv` with columns like `id`, `name`, `host_name`, `neighbourhood`, `room_type`, `price`, `availability_365`, and more.

The dataset is a good fit for this workshop because:

- It has **mixed types** — identifiers, text, dates, categories, numbers
- It has **missing values** with different meanings per column
- It comes in **multiple files** with the same schema (one per city), so we can test multi-file processing

You can download additional cities from [insideairbnb.com/get-the-data](http://insideairbnb.com/get-the-data/) and drop them into the `Airbnb/` folder.

---

## The Tool: LangGraph

**LangGraph** models workflows as directed graphs — nodes are functions, edges are transitions, and state flows through the entire graph.

We chose LangGraph for this workshop because it makes the key patterns easy to demonstrate:

- **State machines** with typed state
- **Conditional branching** based on runtime values
- **Interrupt/resume** for human-in-the-loop
- **Checkpointing** for pause and resume across sessions

> This workshop is **not a LangGraph promotion**. The patterns we cover - state machines, interrupt/resume, conditional routing — apply to any agent framework. LangGraph is simply a convenient tool to teach them.

### Core Concepts

| Concept | What it is |
|---------|-----------|
| **StateGraph** | A container that holds nodes and edges |
| **State** | A `TypedDict` that flows through every node |
| **Node** | A Python function: receives state, returns partial updates |
| **Edge** | A fixed connection from one node to the next |
| **Conditional edge** | A routing function that picks the next node based on state |
| **`interrupt()`** | Pauses execution, saves state, returns control to the caller |
| **`Command(resume=...)`** | Resumes execution with user input after an interrupt |
| **MemorySaver** | Checkpoints state after every node so we can resume |

Here is a minimal example — two nodes connected by edges:

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class MyState(TypedDict):
    base_number: int
    increment_by: int
    multiply_by: int
    result: int
    
def increment(state: MyState) -> dict:
    return {"result": state["base_number"] + state["increment_by"]}

def multiply(state: MyState) -> dict:
    return {"result": state["result"] * state["multiply_by"]}

builder = StateGraph(MyState)
builder.add_node("increment", increment)
builder.add_node("multiply", multiply)
builder.add_edge(START, "increment")
builder.add_edge("increment", "multiply")
builder.add_edge("multiply", END)

graph = builder.compile()
result = graph.invoke({"base_number": 0, "increment_by": 2, "multiply_by": 2})
print(result)

---

## The Agent Architecture

This is the full graph we are building:

![Agent Graph](graph.png)

Two phases:

- **Data cleaning** — per file, with loops for column inference, nil handling, and user feedback
- **Analysis & visualization** — linear, across all cleaned files

Let's go through the building blocks one by one.

---

## State: What the Agent Knows

Everything the agent knows lives in a single `TypedDict` called `CombinedAgentState`. Each node reads from this state and returns a partial update — LangGraph merges it back automatically.

The state is organized into sections:

```python
class CombinedAgentState(TypedDict):
    # Multi-file processing
    files_to_process: list[str]
    current_file_index: int
    file_states: dict[str, FileState]    # per-file data and column info

    # Decision storage (for "apply to all files")
    apply_decisions_to_all_files: bool
    stored_column_decisions: dict[str, dict]
    stored_duplicate_decision: bool | None
    stored_nil_decisions: dict[str, str]
    stored_columns_to_drop: list[str]

    # Processing log (append-only via Annotated[list, operator.add])
    processing_log: Annotated[list[str], operator.add]

    # Status tracking
    status: str

    # Analysis & visualization
    analysis_task: AnalysisTask | None
    analysis_results: dict | None
    visualization_task: VisualizationTask | None
    visualization_figure: str | None
```

Each file being processed has its own `FileState`:

```python
class FileState(TypedDict):
    file_path: str
    original_df: pd.DataFrame | None
    working_df: pd.DataFrame | None
    column_names: list[str]
    current_column_index: int
    column_info: dict[str, ColumnInfo]    # per-column inference results
    status: str
```

And each column carries a `ColumnInfo`:

```python
class ColumnInfo(TypedDict):
    column_name: str
    inferred_type: str          # "integer", "float", "datetime", "categorical", ...
    reasoning: str
    sample_values: list[str]
    confidence: float           # 0.0 – 1.0
    user_approved: bool | None  # None = not yet reviewed
    user_correction: str | None
    transformation_applied: bool
```

The full definitions live in `data_agent/states.py`. The key insight: **nodes don't need to know about each other** — they only need to know which state fields to read and which to update.

---

## Nodes, Edges, and the LLM

### Nodes are plain Python functions

Every node follows the same pattern:

```python
def node_name(state: CombinedAgentState) -> dict:
    # read from state
    # do work
    # return a partial dict — LangGraph merges it back
```

No decorators, no base classes — just functions.

### The LLM as an interpreter

The LLM does **not** drive the flow — LangGraph does. The LLM is called *inside* nodes to interpret ambiguous input: user responses, column types, nil-value strategies.

We get structured output using Pydantic schemas:

```python
class UserResponseInterpretation(BaseModel):
    action: Literal["approve", "correct", "skip", "show_more_samples"]
    target_type: str | None = None
    reasoning: str

structured_llm = llm.with_structured_output(UserResponseInterpretation)
result = structured_llm.invoke(prompt)   # returns a Pydantic object, not free text
```

This pattern is used for every LLM call: type inference, duplicate handling, nil values, analysis, visualization.

### Importing the agent's nodes

All node implementations live in the `data_agent/` package. Let's import them:

In [ ]:
import json
from pathlib import Path

import pandas as pd
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
from langgraph.types import Command, interrupt

from data_agent.states import CombinedAgentState, ColumnInfo, FileState
from data_agent.llm_setup import llm
from data_agent.helper_functions import interpret_user_response

from data_agent.cleaning_nodes import (
    load_file_data,                 # Load CSV/PKL into state
    infer_current_column_type,      # Rule-based + LLM type inference
    apply_transformations,          # Apply user-chosen typetransformations
    check_duplicates,               # Detect and optionally drop duplicates
    handle_nil_values,              # Per-column nil strategy via LLM
    show_column_statistics,         # Stats summary, optional column drop
)
from data_agent.analysis_nodes import (
    generate_analysis_prompt,       # Build domain-specific prompt from data
    decide_analysis_task,           # LLM proposes multiple analysis options for user to choose from
    choose_analysis_task,           # User chooses analysis task
    execute_analysis,               # Run chosen analysis across files
)
from data_agent.visualization_nodes import (
    decide_visualization,           # LLM proposes visualization options based on analysis results
    choose_visualization,           # User chooses one visualization
    create_visualization,           # Generate visualization
    export_results,                 # Save and display results
)

print("All imports successful!")

---

## Routing: Controlling the Flow

**Routing functions** sit on conditional edges. They determine which node to execute next based on the current state.

```
                    ┌─────────────────┐
                    │  infer_type     │
                    └────────┬────────┘
                             │
                    ┌────────▼────────┐
                    │ route_after_    │
                    │   inference     │
                    └────────┬────────┘
                             │
           ┌─────────────────┼─────────────────┐
           │                 │                 │
           ▼                 ▼                 ▼
    ┌─────────────┐   ┌─────────────┐   ┌─────────────┐
    │ ask_feedback│   │ apply_trans │   │    END      │
    └─────────────┘   └─────────────┘   └─────────────┘
    (needs user)      (auto-approve)    (error/done)
```

Key rules:

1. **Return type is `Literal`** — return values must match node names exactly
2. **Read state, don't modify** — routing functions are pure readers
3. **Used with `add_conditional_edges`**, not `add_edge`

```python
builder.add_conditional_edges(
    "infer_type",            # from this node
    route_after_inference,   # use this function to decide
    {                        # map return values to node names
        "ask_feedback": "ask_feedback",
        "infer_type": "infer_type",
        "apply_transforms": "apply_transforms",
    }
)
```

Our agent needs three routing functions:

In [ ]:
from typing import Literal

def route_after_inference(state: CombinedAgentState) -> Literal[
    'ask_feedback', 'apply_transforms', 'infer_type'
]:
    """
    Router: Decide whether to ask for feedback, continue inferring, or apply transformations.
    Works with multi-file state structure.
    """
    current_file_idx = state.get('current_file_index', 0)
    files_to_process = state['files_to_process']

    if current_file_idx >= len(files_to_process):
        return 'apply_transforms'

    file_path = files_to_process[current_file_idx]
    file_state = state['file_states'][file_path]

    current_col_idx = file_state['current_column_index']
    total_columns = len(file_state['column_names'])

    if current_col_idx >= total_columns:
        return 'apply_transforms'

    if state['status'] == 'awaiting_feedback':
        return 'ask_feedback'
    else:
        return 'infer_type'


def route_after_nil_values(
    state: CombinedAgentState
) -> Literal['handle_nil_values', 'show_statistics']:
    """
    Router: After handling one nil column, either continue to next nil column or show statistics.
    """
    if state['status'] == 'checking_nil_values':
        return 'handle_nil_values'
    else:
        return 'show_statistics'


def route_after_statistics(
    state: CombinedAgentState
) -> Literal['load_data', 'generate_analysis_prompt']:
    """
    Router: After showing statistics and column drop, either load next file or start analysis.
    """
    if state['status'] == 'complete':
        return 'generate_analysis_prompt'
    else:
        return 'load_data'

---

## Human-in-the-Loop: `interrupt()` and `Command(resume=...)` Pattern

```
┌──────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
│  Node A  │ ──► │  Node B  │ ──► │  PAUSE   │ ◄── │ interrupt()
└──────────┘     └──────────┘     └──────────┘     └──────────┘
                                       │
                                       ▼
                               [ User provides input ]
                                       │
                                       ▼
                               ┌──────────────────┐
                               │ Command(resume=) │
                               └──────────────────┘
                                       │
                                       ▼
                               ┌──────────┐
                               │  Node C  │ ──► ...
                               └──────────┘
```

This is the core pattern that makes the agent interactive.

When `interrupt()` is called, the graph **saves its state** and **returns control** to the caller.The caller collects user input and resumes with `Command(resume=value)`.

```python
def my_node(state):
    user_input = interrupt({"prompt": "What should I do?"})
    # execution resumes here after Command(resume=...) is sent
    return {"user_choice": user_input}
```

The `ask_user_feedback` node uses this pattern. For each column where the agent's confidence is below 80%, it:

1. Presents the inference (type, confidence, sample values)
2. Calls `interrupt()` to pause
3. When resumed, passes the user's natural-language response to the LLM for interpretation
4. Updates the state with the decision

For subsequent files, if the user chose "apply to all", stored decisions are auto-applied without interrupting.

In [ ]:
def ask_user_feedback(state: CombinedAgentState) -> dict:
    """
    Node: Present the inference to the user and ask for feedback.
    Uses LangGraph's interrupt() to pause and wait for user input.
    Uses LLM to interpret user responses. Auto-applies stored decisions for subsequent files.
    """
    current_file_idx = state.get('current_file_index', 0)
    files_to_process = state['files_to_process']
    file_path = files_to_process[current_file_idx]
    file_state = state['file_states'][file_path]
    apply_to_all = state.get('apply_decisions_to_all_files', False)

    column_names = file_state['column_names']
    current_col_idx = file_state['current_column_index']
    column_name = column_names[current_col_idx]
    col_info = file_state['column_info'][column_name]

    # Check if we should auto-apply a stored decision
    decisions_preloaded = state.get('decisions_preloaded', False)
    auto_apply = False
    stored_decision = None
    if (apply_to_all and current_file_idx > 0) or decisions_preloaded:
        stored_column_decisions = state.get('stored_column_decisions', {})
        if column_name in stored_column_decisions:
            stored_decision = stored_column_decisions[column_name]
            auto_apply = True
            print(f"\nAuto-applying stored decision for column '{column_name}':")
            print(f"   Type: {stored_decision.get('type')}")
            print(f"   Approved: {stored_decision.get('approved')}")

    if not auto_apply:
        num_samples = state.get('_show_samples_count', 5)
        samples_to_show = col_info['sample_values'][:num_samples]

        file_name = Path(file_path).name
        inferred_type_upper = col_info['inferred_type'].upper()
        confidence_pct = f"{col_info['confidence']:.1%}"

        message = (
            f"\n{'='*65}\n"
            f"  FILE {current_file_idx + 1}/{len(files_to_process)}: {file_name}\n"
            f"  COLUMN {current_col_idx + 1} of {len(column_names)}: {column_name}\n"
            f"{'='*65}\n"
            f"  Inferred Type: {inferred_type_upper}\n"
            f"  Confidence: {confidence_pct}\n"
            f"  Reasoning: {col_info['reasoning']}\n"
            f"  Sample Values ({len(samples_to_show)} shown): {samples_to_show}\n"
            f"{'='*65}\n"
            f"  OPTIONS: approve / correct to <type> / skip / show more samples\n"
        )
        print(message)

        # INTERRUPT -- pause the graph and wait for user input
        user_response = interrupt({
            'file_path': file_path,
            'column_name': column_name,
            'inferred_type': col_info['inferred_type'],
            'confidence': col_info['confidence'],
            'prompt': f"What type should '{column_name}' be? (or approve '{col_info['inferred_type']}')"
        })

        print(f"\nInterpreting your response: '{user_response}'...")
        interpretation = interpret_user_response(
            str(user_response), column_name, col_info['inferred_type'], col_info['sample_values']
        )

        print(f"   Action: {interpretation.action}")
        print(f"   Reasoning: {interpretation.reasoning}")

        if interpretation.action == 'show_more_samples':
            new_count = min(num_samples + 5, len(col_info['sample_values']))
            print(f"\nShowing {new_count} samples...")
            return {'_show_samples_count': new_count, 'status': 'awaiting_feedback'}

        user_approved = interpretation.action == 'approve'
        user_correction = None
        if interpretation.action == 'correct' and interpretation.target_type:
            user_correction = interpretation.target_type
        elif interpretation.action == 'skip':
            user_correction = 'skip'
    else:
        user_approved = stored_decision.get('approved', False)
        user_correction = stored_decision.get('correction')

    updated_file_states = dict(state['file_states'])
    updated_file = dict(updated_file_states[file_path])
    updated_column_info = dict(updated_file['column_info'])
    updated_col = dict(updated_column_info[column_name])
    updated_col['user_approved'] = user_approved
    updated_col['user_correction'] = user_correction
    updated_column_info[column_name] = ColumnInfo(**updated_col)
    updated_file['column_info'] = updated_column_info
    updated_file['current_column_index'] = current_col_idx + 1
    updated_file_states[file_path] = FileState(**updated_file)

    return {
        'file_states': updated_file_states,
        'status': 'inferring',
        '_show_samples_count': 5
    }

print("ask_user_feedback node defined!")

---

## Building the Graph

`build_graph()` wires all the pieces together:

- `add_node(name, function)` — register a node
- `add_edge(source, target)` — unconditional transition
- `add_conditional_edges(source, router_fn, mapping)` — branching based on state

The mapping dict in `add_conditional_edges` maps the router's return value to the target node name.

In [ ]:
def build_graph():
    builder = StateGraph(CombinedAgentState)

    # Data Cleaning Phase
    builder.add_node("load_data", load_file_data)
    builder.add_node("infer_type", infer_current_column_type)
    builder.add_node("ask_feedback", ask_user_feedback)
    builder.add_node("apply_transforms", apply_transformations)
    builder.add_node("check_duplicates", check_duplicates)
    builder.add_node("handle_nil_values", handle_nil_values)
    builder.add_node("show_statistics", show_column_statistics)

    # Analysis & Visualization Phase
    builder.add_node("generate_analysis_prompt", generate_analysis_prompt)
    builder.add_node("decide_analysis", decide_analysis_task)
    builder.add_node("choose_analysis", choose_analysis_task)
    builder.add_node("execute_analysis", execute_analysis)
    builder.add_node("decide_visualization", decide_visualization)
    builder.add_node("choose_visualization", choose_visualization)
    builder.add_node("create_visualization", create_visualization)
    builder.add_node("export_results", export_results)

    # Edges: Data Cleaning
    builder.add_edge(START, "load_data")
    builder.add_edge("load_data", "infer_type")

    builder.add_conditional_edges(
        "infer_type",
        route_after_inference,
        {
            'ask_feedback': 'ask_feedback',
            'infer_type': 'infer_type',
            'apply_transforms': 'apply_transforms',
        }
    )

    builder.add_conditional_edges(
        "ask_feedback",
        route_after_inference,
        {
            'ask_feedback': 'ask_feedback',
            'infer_type': 'infer_type',
            'apply_transforms': 'apply_transforms',
        }
    )

    builder.add_edge("apply_transforms", "check_duplicates")
    builder.add_edge("check_duplicates", "handle_nil_values")

    builder.add_conditional_edges(
        "handle_nil_values",
        route_after_nil_values,
        {
            'handle_nil_values': 'handle_nil_values',
            'show_statistics': 'show_statistics',
        }
    )

    builder.add_conditional_edges(
        "show_statistics",
        route_after_statistics,
        {
            'load_data': 'load_data',
            'generate_analysis_prompt': 'generate_analysis_prompt',
        }
    )

    # Edges: Analysis & Visualization
    builder.add_edge("generate_analysis_prompt", "decide_analysis")
    builder.add_edge("decide_analysis", "choose_analysis")
    builder.add_edge("choose_analysis", "execute_analysis")
    builder.add_edge("execute_analysis", "decide_visualization")
    builder.add_edge("decide_visualization", "choose_visualization")
    builder.add_edge("choose_visualization", "create_visualization")
    builder.add_edge("create_visualization", "export_results")
    builder.add_edge("export_results", END)

    return builder

print("build_graph function defined!")

---

## Memory and Checkpointing

`interrupt()` only works if the graph has a **checkpointer**. Without one, there is no way to save and restore state between pauses.

**Checkpointing** allows the agent to:
1. **Save state** after each node execution
2. **Resume** from where it left off after `interrupt()`
3. **Inspect** current state with `get_state()`

```
┌─────────────────────────────────────────────────────────────────────────┐
│                          CHECKPOINTING ARCHITECTURE                     │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│    ┌─────────────┐      ┌─────────────┐      ┌─────────────┐            │
│    │   Node A    │ ──►  │   Node B    │ ──►  │   Node C    │            │
│    └─────────────┘      └─────────────┘      └──────┬──────┘            │
│           │                    │                    │                   │
│           ▼                    ▼                    ▼                   │
│    ┌─────────────────────────────────────────────────────────┐          │
│    │                   MemorySaver                           |          |
|    |                                                         │          │
│    │   checkpoint_1    checkpoint_2    checkpoint_3   ...    │          │
│    └─────────────────────────────────────────────────────────┘          │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Key Components:

| Component | Purpose |
|-----------|----------|
| `MemorySaver` | In-memory checkpointer (for development) |
| `JsonPlusSerializer` | Handles complex objects like DataFrames |
| `thread_id` | Identity of a conversation/run |
| `config` | Passed to every graph invocation |

In [ ]:
# Create the checkpointer with JSON serialization (supports DataFrames via pickle fallback)
memory = MemorySaver(serde=JsonPlusSerializer(pickle_fallback=True))

# Compile the graph with the checkpointer
graph = build_graph().compile(checkpointer=memory)

print("Graph compiled with checkpointer!")
print(f"Checkpointer type: {type(memory).__name__}")

In [ ]:
# Create a config with a unique thread_id
# This identifies "this conversation" for checkpointing
import uuid

thread_id = "workshop_run_" + str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print(f"Thread ID: {thread_id}")
print(f"Config: {config}")

### Why `thread_id` matters:

- Same `thread_id` = **resume** the conversation
- Different `thread_id` = **start fresh**

In production, you'd typically use:
- User session ID
- Database record ID
- Or any unique identifier for the "conversation"

---

## The Execution Loop

The caller runs the graph in a `while True` loop:

```python
while True:
    async for chunk in graph.astream(current_input, config=config, stream_mode='updates'):
        if '__interrupt__' in chunk:
            break
    user_input = input("Your response: ")
    current_input = Command(resume=user_input)
```

When the graph hits `interrupt()`, we need to:
1. **Detect** the interrupt
2. **Get user input** (from terminal, UI, etc.)
3. **Resume** with `Command(resume=user_input)`

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         EXECUTION LOOP                                  │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  1. INVOKE:   graph.astream(input, config)                              │
│                                                                         │
│  2. PROCESS:  Stream events until interrupt or completion               │
│                                                                         │
│  3. DETECT:   if '__interrupt__' in event:                              │
│                   # Graph has paused for user input                     │
│                   interrupt_data = event['__interrupt__']               │
│                                                                         │
│  4. PROMPT:   user_input = input(prompt_from_interrupt_data)            │
│                                                                         │
│  5. RESUME:   graph.astream(Command(resume=user_input), config)         │
│                                                                         │
│  6. REPEAT:   Go back to step 2                                         │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

Let's set up the files and run the agent.

In [ ]:
FILE_PATHS = [
    "Airbnb/Barcelona/listings.csv",
    "Airbnb/Athens/listings.csv",
    "Airbnb/Berlin/listings.csv",
]

existing_files = [f for f in FILE_PATHS if Path(f).exists()]

if len(existing_files) == 0:
    print("No files found to process. Please check the file paths and try again.")
else:
    print(f"Files to process: {len(existing_files)}")
    for i, fp in enumerate(existing_files):
        print(f"  {i+1}. {fp}")

In [ ]:
initial_state = {
    'files_to_process': existing_files,
    'current_file_index': 0,
    'status': 'initializing',
    'file_states': {},
    'processing_log': [],
    'apply_decisions_to_all_files': False,
    'decisions_preloaded': False,
    'stored_column_decisions': {},
    'stored_duplicate_decision': None,
    'stored_nil_decisions': {},
    'stored_columns_to_drop': [],
    'analysis_options': [],
    'analysis_task': {},
    'analysis_results': {},
    'insights': [],
    'summary': '',
    'visualization_options': [],
    'visualization_task': {},
    'visualization_figure': None,
}


print("Initial state created!")

### Optional: You can load saved decisions

In [ ]:
from data_agent.helper_functions import load_saved_decisions
DECISIONS_FILE = 'cleaning_decisions.json'

loaded_state, loaded_config = load_saved_decisions(DECISIONS_FILE, existing_files)
if loaded_state is not None:
    initial_state = loaded_state
    config = loaded_config

---

## Running the Agent

In [ ]:
print("=" * 70)
print("STARTING DATA AGENT")
print("=" * 70)
print(f"\nFiles to process: {len(existing_files)}")
print("The agent will pause at decision points for your input.")
print("Type 'quit' or 'exit' to stop early.\n")

current_input = initial_state

while True:
    interrupted = False

    async for chunk in graph.astream(
        current_input,
        stream_mode='updates',
        config=config
    ):
        if '__interrupt__' in chunk:
            interrupted = True
            break
        else:
            if chunk:
                node_name = list(chunk.keys())[0]

    if not interrupted:
        state = graph.get_state(config)
        if not state.next:
            print("\n" + "=" * 70)
            print("All processing complete!")
            break
        else:
            print(f"Graph paused unexpectedly. Next nodes: {state.next}")
            break

    print("\n" + "-" * 50)
    user_input = input("Your response: ")

    if user_input.lower() in ['quit', 'exit', 'q']:
        print("\nAgent stopped by user.")
        break

    current_input = Command(resume=user_input)

---

## Inspecting State and Saving Decisions

After the agent completes (or you stop it early), you can inspect the current state and save your decisions for reuse in future runs.

In [ ]:
state = graph.get_state(config)

print("CURRENT AGENT STATE")
print("=" * 50)
print(f"Status: {state.values.get('status', 'N/A')}")
print(f"Current file index: {state.values.get('current_file_index', 0)}")
print(f"Files to process: {state.values.get('files_to_process', [])}")
print(f"Apply to all files: {state.values.get('apply_decisions_to_all_files', False)}")

file_states = state.values.get('file_states', {})
if file_states:
    print(f"\nFiles loaded: {len(file_states)}")
    for fp, fs in file_states.items():
        df = fs.get('working_df')
        rows = len(df) if df is not None else 'N/A'
        print(f"  {Path(fp).name}: {rows} rows, status={fs.get('status', 'N/A')}")

processing_log = state.values.get('processing_log', [])
if processing_log:
    print(f"\nProcessing log ({len(processing_log)} entries):")
    for log in processing_log[-5:]:
        print(f"  {log}")

if state.tasks:
    print(f"\nWaiting for input: Yes")
else:
    print(f"\nWaiting for input: No")

In [ ]:
current_state = graph.get_state(config).values

decisions = {
    'stored_column_decisions': current_state.get('stored_column_decisions', {}),
    'stored_duplicate_decision': current_state.get('stored_duplicate_decision'),
    'stored_nil_decisions': current_state.get('stored_nil_decisions', {}),
    'stored_columns_to_drop': current_state.get('stored_columns_to_drop', []),
}

DECISIONS_FILE = 'cleaning_decisions.json'
with open(DECISIONS_FILE, 'w') as f:
    json.dump(decisions, f, indent=2)

print(f"Decisions saved to: {DECISIONS_FILE}")
print(f"  Column type decisions: {len(decisions['stored_column_decisions'])} columns")
dup_action = 'Drop' if decisions['stored_duplicate_decision'] else 'Keep'
print(f"  Duplicate handling: {dup_action}")
print(f"  Nil value handling: {len(decisions['stored_nil_decisions'])} columns")
print(f"  Columns to drop: {decisions['stored_columns_to_drop'] or 'None'}")

---

## Recap

We started with a problem: manual data cleaning doesn't scale, but fully automated scripts throw away human judgment.

We solved it by building an agent that **automates the routine** and **pauses for the human** when judgment matters.

| Pattern | How we used it |
|---------|---------------|
| **State machine** | `CombinedAgentState` flows through every node — each reads and writes only what it needs |
| **Nodes as functions** | Plain Python functions, no framework magic — `def node(state) → dict` |
| **Conditional routing** | `route_after_inference`, `route_after_nil_values` — loops and branching driven by `status` |
| **Human-in-the-loop** | `interrupt()` pauses, `Command(resume=...)` continues — the human stays in control |
| **LLM as interpreter** | Structured output with Pydantic — the LLM reads ambiguity, it doesn't drive the flow |
| **Checkpointing** | `MemorySaver` saves state after every node — stop and resume anytime |

### Where to go from here

- **Add more cities** — drop any Inside Airbnb CSV into `Airbnb/` and the agent picks it up
- **Swap the LLM** — change one line in `llm_setup.py` to use a different model or provider
- **Add new nodes** — outlier detection, correlation analysis, export to a database
- **Move to production** — replace `MemorySaver` with a SQL-backed checkpointer (e.g. `SqliteSaver` or `PostgresSaver`) for persistent state across sessions, and add a web UI instead of `input()`

The LLM handles ambiguity. The graph handles flow. The human stays in control.

---

## Questions?